In [1]:
import json
import re
import logging
import torch
import spacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm
from datetime import datetime
import os

# Configuration
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
SPACY_MODEL = "en_core_web_sm"  # Download with: python -m spacy download en_core_web_sm
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_QUANTIZATION = True

# File Paths
NER_OUTPUT_PATH = "extracted_entities_structured.json"
GRAMMAR_ANALYSIS_OUTPUT = "grammar_analysis_results.json"
GRAMMAR_RELATIONS_OUTPUT = "grammar_aware_relations.json"
GRAMMAR_LABEL_STUDIO_OUTPUT = "grammar_label_studio.json"

# Processing Configuration
MIN_CONFIDENCE_THRESHOLD = 0.5
ENABLE_GRAMMAR_ANALYSIS = True
ENABLE_CLAUSE_SPLITTING = True
MAX_ENTITIES_PER_PROMPT = 10

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("grammar_aware_extraction.log"),
        logging.StreamHandler()
    ]
)

print("🔧 Grammar-Aware Configuration loaded successfully")
print(f"📊 Device: {DEVICE}")
print(f"🔍 SpaCy Model: {SPACY_MODEL}")
print(f"🧠 Grammar Analysis: {ENABLE_GRAMMAR_ANALYSIS}")
print(f"📝 Clause Splitting: {ENABLE_CLAUSE_SPLITTING}")

🔧 Grammar-Aware Configuration loaded successfully
📊 Device: cuda
🔍 SpaCy Model: en_core_web_sm
🧠 Grammar Analysis: True
📝 Clause Splitting: True


In [2]:
def load_spacy_model():
    """Load SpaCy model for grammatical analysis."""
    try:
        nlp = spacy.load(SPACY_MODEL)
        print(f"✅ SpaCy model '{SPACY_MODEL}' loaded successfully")
        print(f"📋 Pipeline components: {nlp.pipe_names}")
        return nlp
    except OSError:
        print(f"❌ SpaCy model '{SPACY_MODEL}' not found!")
        print("💡 Install it with: python -m spacy download en_core_web_sm")
        raise

def analyze_sentence_grammar(sentence, nlp):
    """Comprehensive grammatical analysis of a sentence."""
    
    doc = nlp(sentence)
    
    analysis = {
        'sentence': sentence,
        'tokens': [],
        'entities_spacy': [],
        'noun_phrases': [],
        'verb_phrases': [],
        'clauses': [],
        'dependencies': [],
        'sentence_structure': {},
        'subjects': [],
        'objects': [],
        'predicates': []
    }
    
    # Token-level analysis
    for token in doc:
        token_info = {
            'text': token.text,
            'lemma': token.lemma_,
            'pos': token.pos_,
            'tag': token.tag_,
            'dep': token.dep_,
            'head': token.head.text,
            'head_pos': token.head.pos_,
            'children': [child.text for child in token.children],
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'is_punct': token.is_punct,
            'idx': token.idx
        }
        analysis['tokens'].append(token_info)
    
    # SpaCy named entities
    for ent in doc.ents:
        ent_info = {
            'text': ent.text,
            'label': ent.label_,
            'start_char': ent.start_char,
            'end_char': ent.end_char,
            'start_token': ent.start,
            'end_token': ent.end
        }
        analysis['entities_spacy'].append(ent_info)
    
    # Noun phrases
    for chunk in doc.noun_chunks:
        np_info = {
            'text': chunk.text,
            'root': chunk.root.text,
            'root_dep': chunk.root.dep_,
            'root_head': chunk.root.head.text,
            'start_char': chunk.start_char,
            'end_char': chunk.end_char
        }
        analysis['noun_phrases'].append(np_info)
    
    # Find subjects, objects, and predicates
    subjects, objects, predicates = extract_grammatical_roles(doc)
    analysis['subjects'] = subjects
    analysis['objects'] = objects
    analysis['predicates'] = predicates
    
    # Dependency relations
    for token in doc:
        if token.dep_ != 'ROOT':
            dep_info = {
                'child': token.text,
                'child_pos': token.pos_,
                'relation': token.dep_,
                'head': token.head.text,
                'head_pos': token.head.pos_
            }
            analysis['dependencies'].append(dep_info)
    
    # Clause splitting
    if ENABLE_CLAUSE_SPLITTING:
        analysis['clauses'] = split_into_clauses(doc)
    
    # Sentence structure
    analysis['sentence_structure'] = analyze_sentence_structure(doc)
    
    return analysis

def extract_grammatical_roles(doc):
    """Extract subjects, objects, and predicates from the sentence."""
    
    subjects = []
    objects = []
    predicates = []
    
    for token in doc:
        # Subjects (including compound subjects)
        if token.dep_ in ['nsubj', 'nsubjpass', 'csubj', 'csubjpass']:
            subject_info = {
                'text': token.text,
                'full_phrase': get_full_phrase(token),
                'head_verb': token.head.text,
                'type': token.dep_,
                'start_char': token.idx,
                'end_char': token.idx + len(token.text)
            }
            subjects.append(subject_info)
        
        # Objects (direct and indirect)
        elif token.dep_ in ['dobj', 'iobj', 'pobj', 'dative']:
            object_info = {
                'text': token.text,
                'full_phrase': get_full_phrase(token),
                'head_verb': token.head.text,
                'type': token.dep_,
                'start_char': token.idx,
                'end_char': token.idx + len(token.text)
            }
            objects.append(object_info)
        
        # Predicates (main verbs)
        elif token.pos_ == 'VERB' and token.dep_ in ['ROOT', 'aux', 'auxpass']:
            predicate_info = {
                'text': token.text,
                'lemma': token.lemma_,
                'full_phrase': get_verb_phrase(token),
                'type': token.dep_,
                'tense': get_verb_tense(token),
                'start_char': token.idx,
                'end_char': token.idx + len(token.text)
            }
            predicates.append(predicate_info)
    
    return subjects, objects, predicates

def get_full_phrase(token):
    """Get the full phrase for a token including modifiers."""
    
    # Get all children and descendants
    phrase_tokens = [token]
    
    def add_children(tok):
        for child in tok.children:
            if child.dep_ in ['det', 'amod', 'compound', 'prep', 'poss', 'nummod']:
                phrase_tokens.append(child)
                add_children(child)
    
    add_children(token)
    
    # Sort by position and join
    phrase_tokens.sort(key=lambda x: x.i)
    return ' '.join([t.text for t in phrase_tokens])

def get_verb_phrase(token):
    """Get the full verb phrase including auxiliaries and modifiers."""
    
    phrase_tokens = [token]
    
    # Add auxiliary verbs and modifiers
    for child in token.children:
        if child.dep_ in ['aux', 'auxpass', 'neg', 'advmod']:
            phrase_tokens.append(child)
    
    # Sort by position
    phrase_tokens.sort(key=lambda x: x.i)
    return ' '.join([t.text for t in phrase_tokens])

def get_verb_tense(token):
    """Determine verb tense from morphological features."""
    
    if token.tag_ in ['VBD', 'VBN']:
        return 'past'
    elif token.tag_ in ['VBZ', 'VBP']:
        return 'present'
    elif token.tag_ in ['VBG']:
        return 'progressive'
    elif 'will' in [child.text.lower() for child in token.children]:
        return 'future'
    else:
        return 'unknown'

def split_into_clauses(doc):
    """Split sentence into clauses based on grammatical structure."""
    
    clauses = []
    current_clause = []
    
    for token in doc:
        # Start new clause on certain conjunctions or relative pronouns
        if (token.dep_ in ['cc', 'mark'] and 
            token.text.lower() in ['and', 'but', 'or', 'because', 'since', 'while', 'although', 'that', 'which', 'who']):
            
            if current_clause:
                clause_text = ' '.join([t.text for t in current_clause])
                clauses.append({
                    'text': clause_text,
                    'type': 'main' if not clauses else 'subordinate',
                    'connector': token.text if token.dep_ in ['cc', 'mark'] else None
                })
                current_clause = []
        
        current_clause.append(token)
    
    # Add final clause
    if current_clause:
        clause_text = ' '.join([t.text for t in current_clause])
        clauses.append({
            'text': clause_text,
            'type': 'main' if not clauses else 'continuation'
        })
    
    return clauses

def analyze_sentence_structure(doc):
    """Analyze overall sentence structure."""
    
    structure = {
        'sentence_type': 'unknown',
        'main_verb': None,
        'complexity': 'simple',
        'passive_voice': False,
        'negation': False,
        'question': False
    }
    
    # Find main verb (ROOT)
    for token in doc:
        if token.dep_ == 'ROOT':
            structure['main_verb'] = {
                'text': token.text,
                'lemma': token.lemma_,
                'pos': token.pos_
            }
            break
    
    # Check for passive voice
    for token in doc:
        if token.dep_ == 'auxpass':
            structure['passive_voice'] = True
            break
    
    # Check for negation
    for token in doc:
        if token.dep_ == 'neg':
            structure['negation'] = True
            break
    
    # Check if question
    if doc[-1].text == '?':
        structure['question'] = True
        structure['sentence_type'] = 'interrogative'
    elif structure['main_verb'] and structure['main_verb']['pos'] == 'VERB':
        structure['sentence_type'] = 'declarative'
    
    # Determine complexity
    subordinate_markers = ['because', 'since', 'although', 'while', 'that', 'which', 'who']
    if any(token.text.lower() in subordinate_markers for token in doc):
        structure['complexity'] = 'complex'
    elif any(token.dep_ == 'cc' for token in doc):
        structure['complexity'] = 'compound'
    
    return structure

# Load SpaCy model
nlp = load_spacy_model()

print("✅ Grammar analysis functions loaded")

✅ SpaCy model 'en_core_web_sm' loaded successfully
📋 Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
✅ Grammar analysis functions loaded


In [3]:
def integrate_entities_with_grammar(original_entities, grammar_analysis, sentence):
    """Integrate original NER entities with grammatical analysis."""
    
    integrated_entities = []
    
    for entity in original_entities:
        # Start with original entity data
        enhanced_entity = entity.copy()
        
        # Find grammatical role
        grammatical_role = find_entity_grammatical_role(entity, grammar_analysis)
        enhanced_entity['grammatical_role'] = grammatical_role
        
        # Find syntactic head
        syntactic_info = find_entity_syntactic_info(entity, grammar_analysis)
        enhanced_entity['syntactic_info'] = syntactic_info
        
        # Find related phrases
        related_phrases = find_related_phrases(entity, grammar_analysis)
        enhanced_entity['related_phrases'] = related_phrases
        
        # Find clause membership
        clause_info = find_entity_clause(entity, grammar_analysis)
        enhanced_entity['clause_info'] = clause_info
        
        integrated_entities.append(enhanced_entity)
    
    # Add additional entities from grammar analysis
    additional_entities = discover_additional_entities(original_entities, grammar_analysis, sentence)
    integrated_entities.extend(additional_entities)
    
    return integrated_entities

def find_entity_grammatical_role(entity, grammar_analysis):
    """Find the grammatical role of an entity."""
    
    entity_start = entity['start_char']
    entity_end = entity['end_char']
    
    # Check if entity is a subject
    for subject in grammar_analysis['subjects']:
        if (entity_start >= subject['start_char'] and 
            entity_end <= subject['end_char'] + len(subject['full_phrase'])):
            return {
                'role': 'subject',
                'type': subject['type'],
                'head_verb': subject['head_verb'],
                'full_phrase': subject['full_phrase']
            }
    
    # Check if entity is an object
    for obj in grammar_analysis['objects']:
        if (entity_start >= obj['start_char'] and 
            entity_end <= obj['end_char'] + len(obj['full_phrase'])):
            return {
                'role': 'object',
                'type': obj['type'], 
                'head_verb': obj['head_verb'],
                'full_phrase': obj['full_phrase']
            }
    
    # Check if entity is part of a predicate
    for predicate in grammar_analysis['predicates']:
        if (entity_start >= predicate['start_char'] and 
            entity_end <= predicate['end_char'] + len(predicate['full_phrase'])):
            return {
                'role': 'predicate',
                'type': predicate['type'],
                'tense': predicate['tense'],
                'full_phrase': predicate['full_phrase']
            }
    
    # Default to modifier or other
    return {
        'role': 'modifier',
        'type': 'unknown',
        'description': 'Entity does not match major grammatical roles'
    }

def find_entity_syntactic_info(entity, grammar_analysis):
    """Find syntactic information for an entity."""
    
    entity_text = entity['text'].lower()
    
    # Find matching tokens
    matching_tokens = []
    for token in grammar_analysis['tokens']:
        if entity_text in token['text'].lower() or token['text'].lower() in entity_text:
            matching_tokens.append(token)
    
    if not matching_tokens:
        return {'found': False}
    
    # Get info from best matching token
    best_match = max(matching_tokens, key=lambda x: len(x['text']))
    
    return {
        'found': True,
        'pos': best_match['pos'],
        'tag': best_match['tag'],
        'dep': best_match['dep'],
        'head': best_match['head'],
        'head_pos': best_match['head_pos'],
        'children': best_match['children'],
        'lemma': best_match['lemma']
    }

def find_related_phrases(entity, grammar_analysis):
    """Find noun phrases and other phrases related to the entity."""
    
    related = []
    entity_text = entity['text'].lower()
    
    # Check noun phrases
    for np in grammar_analysis['noun_phrases']:
        if (entity_text in np['text'].lower() or 
            any(word in np['text'].lower() for word in entity_text.split())):
            related.append({
                'type': 'noun_phrase',
                'text': np['text'],
                'root': np['root'],
                'root_dep': np['root_dep']
            })
    
    return related

def find_entity_clause(entity, grammar_analysis):
    """Find which clause(s) the entity belongs to."""
    
    entity_start = entity['start_char']
    entity_end = entity['end_char']
    
    for i, clause in enumerate(grammar_analysis['clauses']):
        clause_start = grammar_analysis['sentence'].find(clause['text'])
        clause_end = clause_start + len(clause['text'])
        
        if entity_start >= clause_start and entity_end <= clause_end:
            return {
                'clause_index': i,
                'clause_text': clause['text'],
                'clause_type': clause['type'],
                'connector': clause.get('connector')
            }
    
    return {'found': False}

def discover_additional_entities(original_entities, grammar_analysis, sentence):
    """Discover additional entities from grammatical analysis."""
    
    existing_texts = {ent['text'].lower() for ent in original_entities}
    additional_entities = []
    
    # Add important subjects that weren't captured
    for subject in grammar_analysis['subjects']:
        if subject['text'].lower() not in existing_texts and len(subject['text']) > 2:
            additional_entities.append({
                'text': subject['text'],
                'label': 'DISCOVERED_SUBJECT',
                'start_char': subject['start_char'],
                'end_char': subject['end_char'],
                'confidence': 'MEDIUM',
                'discovery_method': 'grammar_subject',
                'grammatical_role': {
                    'role': 'subject',
                    'type': subject['type'],
                    'head_verb': subject['head_verb'],
                    'full_phrase': subject['full_phrase']
                }
            })
    
    # Add important objects that weren't captured
    for obj in grammar_analysis['objects']:
        if obj['text'].lower() not in existing_texts and len(obj['text']) > 2:
            additional_entities.append({
                'text': obj['text'],
                'label': 'DISCOVERED_OBJECT',
                'start_char': obj['start_char'],
                'end_char': obj['end_char'],
                'confidence': 'MEDIUM',
                'discovery_method': 'grammar_object',
                'grammatical_role': {
                    'role': 'object',
                    'type': obj['type'],
                    'head_verb': obj['head_verb'],
                    'full_phrase': obj['full_phrase']
                }
            })
    
    # Add important noun phrases for LULC domain
    lulc_keywords = ['forest', 'urban', 'area', 'land', 'cover', 'change', 'growth', 'expansion', 'decline']
    for np in grammar_analysis['noun_phrases']:
        if (np['text'].lower() not in existing_texts and 
            any(keyword in np['text'].lower() for keyword in lulc_keywords) and
            len(np['text']) > 3):
            
            additional_entities.append({
                'text': np['text'],
                'label': 'DISCOVERED_LULC',
                'start_char': np['start_char'],
                'end_char': np['end_char'],
                'confidence': 'LOW',
                'discovery_method': 'grammar_noun_phrase',
                'grammatical_role': {
                    'role': 'noun_phrase',
                    'root': np['root'],
                    'root_dep': np['root_dep']
                }
            })
    
    return additional_entities

print("✅ Entity-grammar integration functions loaded")

✅ Entity-grammar integration functions loaded


In [4]:
# Global variables
model = None
tokenizer = None

def load_mistral_model():
    """Load Mistral model for relation extraction."""
    global model, tokenizer
    
    if model is not None and tokenizer is not None:
        print("✅ Using existing Mistral model")
        return model, tokenizer
    
    print("🔄 Loading Mistral model...")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        if USE_QUANTIZATION:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            print("🔧 Using 4-bit quantization")
        else:
            quantization_config = None
        
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map="auto",
            quantization_config=quantization_config,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        )
        
        print(f"✅ Mistral model loaded on {DEVICE}")
        return model, tokenizer
    
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        raise

def load_and_process_data(file_path):
    """Load NER data and process with grammar analysis."""
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"✅ Loaded {len(data)} sentences from {file_path}")
        
        processed_data = []
        
        for i, sentence_data in enumerate(tqdm(data, desc="Processing with grammar analysis")):
            try:
                sentence = sentence_data['original_sentence']
                original_entities = sentence_data['entities']
                
                # Perform grammatical analysis
                grammar_analysis = analyze_sentence_grammar(sentence, nlp)
                
                # Integrate entities with grammar
                enhanced_entities = integrate_entities_with_grammar(
                    original_entities, grammar_analysis, sentence
                )
                
                processed_item = {
                    'id': sentence_data.get('id', i),
                    'original_sentence': sentence,
                    'original_entities': original_entities,
                    'enhanced_entities': enhanced_entities,
                    'grammar_analysis': grammar_analysis,
                    'processing_metadata': {
                        'original_entity_count': len(original_entities),
                        'enhanced_entity_count': len(enhanced_entities),
                        'entities_discovered': len(enhanced_entities) - len(original_entities),
                        'grammatical_roles_found': len([e for e in enhanced_entities if e.get('grammatical_role', {}).get('role') != 'modifier']),
                        'clauses_found': len(grammar_analysis['clauses'])
                    }
                }
                
                processed_data.append(processed_item)
                
                # Print progress for first few items
                if i < 3:
                    print(f"\n📝 Sample {i+1}:")
                    print(f"   Sentence: {sentence[:80]}...")
                    print(f"   Entities: {len(original_entities)} → {len(enhanced_entities)} (+{len(enhanced_entities) - len(original_entities)})")
                    print(f"   Grammatical roles: {processed_item['processing_metadata']['grammatical_roles_found']}")
                    print(f"   Clauses: {len(grammar_analysis['clauses'])}")
                
            except Exception as e:
                print(f"❌ Error processing sentence {i}: {e}")
                # Add error placeholder
                processed_data.append({
                    'id': sentence_data.get('id', i),
                    'error': str(e),
                    'original_sentence': sentence_data.get('original_sentence', ''),
                    'processing_failed': True
                })
        
        return processed_data
        
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return []

# Load model and process data
model, tokenizer = load_mistral_model()
processed_sentences = load_and_process_data(NER_OUTPUT_PATH)

# Display summary statistics
if processed_sentences:
    successful = [s for s in processed_sentences if not s.get('processing_failed', False)]
    
    print(f"\n📊 PROCESSING SUMMARY:")
    print(f"Total sentences: {len(processed_sentences)}")
    print(f"Successfully processed: {len(successful)}")
    print(f"Failed: {len(processed_sentences) - len(successful)}")
    
    if successful:
        avg_entities_original = np.mean([s['processing_metadata']['original_entity_count'] for s in successful])
        avg_entities_enhanced = np.mean([s['processing_metadata']['enhanced_entity_count'] for s in successful])
        avg_discovered = np.mean([s['processing_metadata']['entities_discovered'] for s in successful])
        avg_roles = np.mean([s['processing_metadata']['grammatical_roles_found'] for s in successful])
        avg_clauses = np.mean([s['processing_metadata']['clauses_found'] for s in successful])
        
        print(f"Average entities: {avg_entities_original:.1f} → {avg_entities_enhanced:.1f}")
        print(f"Average discovered per sentence: {avg_discovered:.1f}")
        print(f"Average grammatical roles found: {avg_roles:.1f}")
        print(f"Average clauses per sentence: {avg_clauses:.1f}")

print("✅ Data processing complete")

🔄 Loading Mistral model...
🔧 Using 4-bit quantization


2025-06-27 14:31:58,955 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Mistral model loaded on cuda
✅ Loaded 2003 sentences from extracted_entities_structured.json


Processing with grammar analysis:   0%|          | 0/2003 [00:00<?, ?it/s]


📝 Sample 1:
   Sentence: Simulation results reveal that the landscape of Thimphu city has changed conside...
   Entities: 5 → 11 (+6)
   Grammatical roles: 11
   Clauses: 3

📝 Sample 2:
   Sentence: The study observed a significant increase (12.77%) in built-up area from 2002 (5...
   Entities: 7 → 13 (+6)
   Grammatical roles: 11
   Clauses: 1

📝 Sample 3:
   Sentence: On the contrary, forest cover declined drastically (15.25%) followed by agricult...
   Entities: 5 → 8 (+3)
   Grammatical roles: 8
   Clauses: 1

📊 PROCESSING SUMMARY:
Total sentences: 2003
Successfully processed: 2003
Failed: 0
Average entities: 4.8 → 12.2
Average discovered per sentence: 7.4
Average grammatical roles found: 10.2
Average clauses per sentence: 2.7
✅ Data processing complete


In [12]:
# Define allowed entity types only
ALLOWED_ENTITY_TYPES = [
    "LULC", "PROCESS", "CHANGE", "SURFACE_UNIT", "COORDINATES", 
    "GPE", "LOC", "DATE", "PERCENT", "QUANTITY", "CARDINAL"
]

# Define LULC-specific relation types only
LULC_RELATION_TYPES = [
    "causes", "leads_to", "results_in", "affects", "impacts",
    "located_in", "within", "encompasses", "adjacent_to", "borders",
    "occurs_during", "happens_in", "temporal_context",
    "comprises", "covers", "measures", "quantifies", "equals",
    "converts_to", "transforms_into", "changes_to", "replaces",
    "increases", "decreases", "expands", "shrinks", "grows"
]


def integrate_entities_with_grammar_fixed(original_entities, grammar_analysis, sentence):
    """Fixed entity integration that only uses allowed entity types."""
    
    integrated_entities = []
    
    # Process original entities with proper classification
    for entity in original_entities:
        enhanced_entity = entity.copy()
        
        # Re-classify entity properly
        grammar_info = find_entity_syntactic_info(entity, grammar_analysis)
        proper_label = classify_entity_properly(entity['text'], grammar_info, sentence)
        enhanced_entity['label'] = proper_label
        
        # Add grammatical information
        grammatical_role = find_entity_grammatical_role(entity, grammar_analysis)
        enhanced_entity['grammatical_role'] = grammatical_role
        enhanced_entity['syntactic_info'] = grammar_info
        enhanced_entity['confidence'] = 'HIGH'  # Original entities get high confidence
        
        integrated_entities.append(enhanced_entity)
    
    # Add important missing entities from grammar analysis
    additional_entities = discover_missing_entities_fixed(original_entities, grammar_analysis, sentence)
    integrated_entities.extend(additional_entities)
    
    # Remove duplicates and filter
    final_entities = remove_duplicate_entities(integrated_entities)
    
    return final_entities

def discover_missing_entities_fixed(original_entities, grammar_analysis, sentence):
    """Discover missing entities using only allowed types."""
    
    existing_texts = {ent['text'].lower() for ent in original_entities}
    missing_entities = []
    
    # Check subjects for important LULC entities
    for subject in grammar_analysis['subjects']:
        if subject['text'].lower() not in existing_texts and len(subject['text']) > 2:
            # Classify properly
            proper_label = classify_entity_properly(subject['text'], {'pos': 'NOUN'}, sentence)
            
            missing_entities.append({
                'text': subject['text'],
                'label': proper_label,
                'start_char': subject['start_char'],
                'end_char': subject['end_char'],
                'confidence': 'MEDIUM',
                'grammatical_role': {
                    'role': 'subject',
                    'type': subject['type'],
                    'head_verb': subject['head_verb']
                }
            })
    
    # Check objects for important entities
    for obj in grammar_analysis['objects']:
        if obj['text'].lower() not in existing_texts and len(obj['text']) > 2:
            proper_label = classify_entity_properly(obj['text'], {'pos': 'NOUN'}, sentence)
            
            missing_entities.append({
                'text': obj['text'],
                'label': proper_label,
                'start_char': obj['start_char'],
                'end_char': obj['end_char'],
                'confidence': 'MEDIUM',
                'grammatical_role': {
                    'role': 'object',
                    'type': obj['type'],
                    'head_verb': obj['head_verb']
                }
            })
    
    # Check for important LULC noun phrases
    for np in grammar_analysis['noun_phrases']:
        if (np['text'].lower() not in existing_texts and 
            len(np['text']) > 3 and
            is_important_lulc_phrase(np['text'])):
            
            proper_label = classify_entity_properly(np['text'], {'pos': 'NOUN'}, sentence)
            
            missing_entities.append({
                'text': np['text'],
                'label': proper_label,
                'start_char': np['start_char'],
                'end_char': np['end_char'],
                'confidence': 'LOW',
                'grammatical_role': {
                    'role': 'noun_phrase',
                    'root': np['root']
                }
            })
    
    return missing_entities

def is_important_lulc_phrase(text):
    """Check if a phrase is important for LULC analysis."""
    
    important_terms = [
        'land use', 'land cover', 'urban area', 'forest area', 'agricultural land',
        'water bodies', 'built up', 'green space', 'natural area', 'developed area',
        'study area', 'research area', 'study period', 'time period'
    ]
    
    text_lower = text.lower()
    return any(term in text_lower for term in important_terms)

def remove_duplicate_entities(entities):
    """Remove duplicate entities and merge similar ones."""
    
    seen_texts = {}
    final_entities = []
    
    for entity in entities:
        text_key = entity['text'].lower().strip()
        
        if text_key not in seen_texts:
            seen_texts[text_key] = entity
            final_entities.append(entity)
        else:
            # Keep the one with higher confidence
            existing = seen_texts[text_key]
            confidence_order = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}
            
            if confidence_order.get(entity.get('confidence', 'LOW'), 1) > \
               confidence_order.get(existing.get('confidence', 'LOW'), 1):
                # Replace with higher confidence entity
                final_entities.remove(existing)
                final_entities.append(entity)
                seen_texts[text_key] = entity
    
    return final_entities

def create_lulc_relation_prompt(sentence, entities, grammar_analysis):
    """Create prompt focused on LULC relations only."""
    
    # Filter entities to reasonable number
    if len(entities) > 8:
        # Prioritize entities with grammatical roles
        prioritized = sorted(entities, 
                           key=lambda x: (
                               x.get('confidence', 'LOW') == 'HIGH',
                               x.get('grammatical_role', {}).get('role') in ['subject', 'object'],
                               len(x['text'])
                           ), reverse=True)
        entities = prioritized[:8]
    
    # Format entities
    entity_list = []
    for i, entity in enumerate(entities):
        role_info = entity.get('grammatical_role', {}).get('role', 'unknown')
        entity_list.append(f"[{i}] {entity['text']} ({entity['label']}) - Role: {role_info}")
    
    entities_str = "\n".join(entity_list)
    
    # Create focused LULC prompt
    prompt = f"""<s>[INST] You are an expert in LULC (Land Use Land Cover) analysis. Extract ONLY meaningful relationships between entities.

**SENTENCE:** {sentence}

**ENTITIES:**
{entities_str}

**CHANGE_TO**: Indicates a direct transformation from one LULC type to another.
- ✅ CORRECT: forest --CHANGE_TO-- cropland (trees cut, land converted to farming)
- ✅ CORRECT: agricultural land --CHANGE_TO-- urban area (farmland developed into city)
- ✅ CORRECT: grassland --CHANGE_TO-- built-up area (grass removed, buildings constructed)
- ❌ WRONG: built-up area --CHANGE_TO-- built-up area (same type, just quantity change)
- ❌ WRONG: forest --CHANGE_TO-- forest (same type, just area change)

**INCREASES_BY/DECREASES_BY**: For quantitative changes within same LULC type
- ✅ CORRECT: built-up area --INCREASES_BY-- 12.77% (more built-up area, not transformation)
- ✅ CORRECT: forest --DECREASES_BY-- 25% (less forest area, not transformation)

**Other Relations:**
- CAUSES: Process entity causes a change (deforestation --CAUSES-- forest loss)
- LOCATED_IN: Spatial relationships (forest --LOCATED_IN-- Brazil)
- OCCURS_DURING: Temporal relationships (change --OCCURS_DURING-- 2018)
- MEASURES: Quantitative relationships (12.77% --MEASURES-- increase)
- AFFECTS: Impact relationships (urbanization --AFFECTS-- forest)
- FROM_TO: Value changes (52.88% --FROM_TO-- 65.5%)
- ENABLES:  process enables another process(deforestation --ENABLES-- urbanization)
**Relationship Types - BE VERY THOUGHTFUL:**

**STRICT RULES:**
1. Use ONLY the entity numbers [0], [1], etc. provided above
2. Use ONLY the allowed relation types listed above
3. Extract ONLY relations that make semantic sense for LULC analysis
4. Focus on meaningful relationships (avoid weak connections)

**OUTPUT FORMAT:**
[source_num] --relation_type--> [target_num] | confidence: HIGH/MEDIUM/LOW

**EXAMPLES:**
[0] --affects--> [1] | confidence: HIGH
[2] --located_in--> [3] | confidence: MEDIUM

**Extract LULC relations:** [/INST]"""
    
    return prompt

def extract_lulc_relations(sentence_data, model, tokenizer):
    """Extract only LULC-relevant relations."""
    
    prompt = create_lulc_relation_prompt(
        sentence_data['original_sentence'],
        sentence_data['enhanced_entities'],
        sentence_data['grammar_analysis']
    )
    
    try:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2500)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.2,  # Lower temperature for more focused output
                do_sample=True,
                top_p=0.8,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.2
            )
        
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        # Parse and validate relations
        relations = parse_and_validate_lulc_relations(response, sentence_data['enhanced_entities'])
        
        return relations
        
    except Exception as e:
        print(f"❌ Error in LULC relation extraction: {e}")
        return []

def parse_and_validate_lulc_relations(response, entities):
    """Parse relations and validate they are LULC-appropriate."""
    
    relations = []
    lines = response.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Parse: [0] --relation--> [1] | confidence: LEVEL
        pattern = r'\[(\d+)\]\s*--([^-]+)-->\s*\[(\d+)\]\s*\|\s*confidence:\s*(\w+)'
        match = re.match(pattern, line)
        
        if match:
            try:
                source_idx = int(match.group(1))
                relation_type = match.group(2).strip()
                target_idx = int(match.group(3))
                confidence = match.group(4).upper()
                
                # Validate indices
                if source_idx >= len(entities) or target_idx >= len(entities):
                    continue
                
                # Validate relation type
                if relation_type not in LULC_RELATION_TYPES:
                    continue
                
                # Validate semantic sense
                if validate_lulc_relation_semantics(entities[source_idx], entities[target_idx], relation_type):
                    relation = {
                        "source": entities[source_idx]['text'],
                        "relationship": relation_type,
                        "target": entities[target_idx]['text'],
                        "confidence": confidence,
                        "source_entity": entities[source_idx],
                        "target_entity": entities[target_idx],
                        "extraction_method": "lulc_focused"
                    }
                    relations.append(relation)
                    
            except (ValueError, IndexError):
                continue
    
    return relations

def validate_lulc_relation_semantics(source_entity, target_entity, relation_type):
    """Validate if a relation makes semantic sense in LULC context."""
    
    source_label = source_entity['label']
    target_label = target_entity['label']
    
    # Define valid relation patterns for LULC
    valid_patterns = {
        'causes': [
            ('PROCESS', 'CHANGE'), ('PROCESS', 'LULC'), ('CHANGE', 'LULC'),
            ('LULC', 'CHANGE'), ('GPE', 'CHANGE')
        ],
        'located_in': [
            ('LULC', 'GPE'), ('LULC', 'LOC'), ('PROCESS', 'GPE'), ('PROCESS', 'LOC')
        ],
        'occurs_during': [
            ('PROCESS', 'DATE'), ('CHANGE', 'DATE'), ('LULC', 'DATE')
        ],
        'quantifies': [
            ('PERCENT', 'LULC'), ('SURFACE_UNIT', 'LULC'), ('CARDINAL', 'LULC'),
            ('QUANTITY', 'LULC')
        ],
        'Transfer_to': [
            ('LULC', 'LULC'),
        ]
    }
    
    # Check if relation type has defined patterns
    if relation_type in valid_patterns:
        return (source_label, target_label) in valid_patterns[relation_type]
    
    # For other relation types, use general validation
    return True

print("✅ Fixed grammar-aware extraction functions loaded")

✅ Fixed grammar-aware extraction functions loaded


In [13]:
def assess_grammar_relation_quality(relations, sentence_data):
    """Assess quality of extracted relations using grammatical information."""
    
    sentence = sentence_data['original_sentence']
    grammar = sentence_data['grammar_analysis']
    
    assessed_relations = []
    
    for relation in relations:
        source_entity = relation['source_entity']
        target_entity = relation['target_entity']
        
        # Calculate quality scores
        grammar_score = calculate_grammar_alignment_score(relation, source_entity, target_entity)
        semantic_score = calculate_semantic_coherence_score(relation, sentence)
        structural_score = calculate_structural_validity_score(relation, grammar)
        
        # Overall quality
        overall_quality = (grammar_score * 0.4 + semantic_score * 0.3 + structural_score * 0.3)
        
        # Enhanced relation with quality metrics
        enhanced_relation = relation.copy()
        enhanced_relation.update({
            'quality_scores': {
                'grammar_alignment': grammar_score,
                'semantic_coherence': semantic_score,
                'structural_validity': structural_score,
                'overall_quality': overall_quality
            },
            'quality_assessment': {
                'high_quality': overall_quality >= 0.7,
                'medium_quality': 0.4 <= overall_quality < 0.7,
                'low_quality': overall_quality < 0.4
            }
        })
        
        assessed_relations.append(enhanced_relation)
    
    return assessed_relations

def calculate_grammar_alignment_score(relation, source_entity, target_entity):
    """Calculate how well the relation aligns with grammatical roles."""
    
    score = 0.5  # Base score
    
    source_role = source_entity.get('grammatical_role', {}).get('role')
    target_role = target_entity.get('grammatical_role', {}).get('role')
    relationship = relation['relationship'].lower()
    
    # Subject-Object relationships (high confidence)
    if source_role == 'subject' and target_role == 'object':
        if any(word in relationship for word in ['causes', 'affects', 'leads', 'results', 'creates']):
            score += 0.3
    
    # Subject-Predicate relationships
    if source_role == 'subject' and 'predicate' in target_role:
        if any(word in relationship for word in ['performs', 'exhibits', 'undergoes']):
            score += 0.2
    
    # Object relationships
    if target_role == 'object':
        if any(word in relationship for word in ['receives', 'affected_by', 'influenced_by']):
            score += 0.2
    
    # Modifier relationships
    if source_role == 'modifier' or target_role == 'modifier':
        if any(word in relationship for word in ['modifies', 'describes', 'qualifies']):
            score += 0.1
    
    # Same clause bonus
    source_clause = source_entity.get('clause_info', {}).get('clause_index')
    target_clause = target_entity.get('clause_info', {}).get('clause_index')
    if source_clause is not None and source_clause == target_clause:
        score += 0.1
    
    return min(score, 1.0)

def calculate_semantic_coherence_score(relation, sentence):
    """Calculate semantic coherence of the relationship."""
    
    score = 0.5
    relationship = relation['relationship'].lower()
    sentence_lower = sentence.lower()
    
    # LULC domain appropriateness
    lulc_terms = ['forest', 'urban', 'land', 'area', 'cover', 'change', 'growth', 'expansion', 'decline']
    source_is_lulc = any(term in relation['source'].lower() for term in lulc_terms)
    target_is_lulc = any(term in relation['target'].lower() for term in lulc_terms)
    
    if source_is_lulc or target_is_lulc:
        score += 0.2
    
    # Relationship type appropriateness
    causal_indicators = ['cause', 'lead', 'result', 'drive', 'trigger']
    temporal_indicators = ['during', 'by', 'in', 'after', 'before']
    spatial_indicators = ['in', 'within', 'at', 'near', 'across']
    
    if any(indicator in relationship for indicator in causal_indicators):
        if any(word in sentence_lower for word in ['due to', 'because', 'caused by', 'leads to']):
            score += 0.2
    
    if any(indicator in relationship for indicator in temporal_indicators):
        if any(word in sentence_lower for word in ['in', 'during', 'by', 'year', 'time']):
            score += 0.2
    
    if any(indicator in relationship for indicator in spatial_indicators):
        if any(word in sentence_lower for word in ['in', 'at', 'within', 'area', 'region']):
            score += 0.2
    
    return min(score, 1.0)

def calculate_structural_validity_score(relation, grammar_analysis):
    """Calculate structural validity based on sentence structure."""
    
    score = 0.5
    
    # Check if relation follows sentence structure
    sentence_structure = grammar_analysis['sentence_structure']
    
    # Passive voice considerations
    if sentence_structure.get('passive_voice'):
        if 'affected_by' in relation['relationship'] or 'influenced_by' in relation['relationship']:
            score += 0.2
    
    # Complex sentence handling
    if sentence_structure.get('complexity') == 'complex':
        # Allow more diverse relationships in complex sentences
        score += 0.1
    
    # Negation handling
    if sentence_structure.get('negation'):
        if any(word in relation['relationship'] for word in ['not', 'lacks', 'without', 'reduces']):
            score += 0.1
    
    return min(score, 1.0)

def filter_high_quality_relations(assessed_relations, min_quality=0.5):
    """Filter relations based on quality scores."""
    
    high_quality = []
    medium_quality = []
    low_quality = []
    
    for relation in assessed_relations:
        quality = relation['quality_scores']['overall_quality']
        
        if quality >= 0.7:
            high_quality.append(relation)
        elif quality >= min_quality:
            medium_quality.append(relation)
        else:
            low_quality.append(relation)
    
    return {
        'high_quality': high_quality,
        'medium_quality': medium_quality,
        'low_quality': low_quality,
        'filtered': high_quality + medium_quality  # Return high and medium quality
    }

print("✅ Quality assessment and validation functions loaded")

✅ Quality assessment and validation functions loaded


In [7]:
def validate_relations_with_context(relations, sentence, context_analysis):
    """Validate relations using context analysis."""
    
    validated_relations = []
    
    for relation in relations:
        # Calculate context compatibility score
        context_score = calculate_context_compatibility(relation, sentence, context_analysis)
        
        # Calculate semantic coherence
        semantic_score = calculate_semantic_coherence(relation)
        
        # Calculate overall quality score
        overall_score = (context_score * 0.6 + semantic_score * 0.4)
        
        if overall_score >= MIN_CONFIDENCE_THRESHOLD:
            relation['context_score'] = context_score
            relation['semantic_score'] = semantic_score
            relation['overall_quality'] = overall_score
            relation['validation_passed'] = True
            
            # Adjust confidence based on scores
            relation['adjusted_confidence'] = adjust_confidence_with_scores(
                relation.get('confidence', 'MEDIUM'),
                overall_score
            )
            
            validated_relations.append(relation)
        else:
            print(f"⚠️ Low quality score ({overall_score:.2f}) for: {relation['source']} -> {relation['target']}")
    
    return validated_relations

def calculate_context_compatibility(relation, sentence, context_analysis):
    """Calculate how well a relation fits the sentence context."""
    
    score = 0.5  # Base score
    
    # Check entity proximity in sentence
    source_pos = sentence.lower().find(relation['source'].lower())
    target_pos = sentence.lower().find(relation['target'].lower())
    
    if source_pos != -1 and target_pos != -1:
        distance = abs(source_pos - target_pos)
        # Closer entities get higher scores
        if distance < 30:
            score += 0.2
        elif distance < 60:
            score += 0.1
        elif distance > 200:
            score -= 0.1  # Penalize very distant entities
    
    # Check relationship type compatibility with context
    rel_type = relation['relationship'].lower()
    
    # Causal relationships should have causal indicators
    if any(causal_word in rel_type for causal_word in ['cause', 'lead', 'result', 'trigger', 'drive']):
        if context_analysis['causal_indicators']:
            score += 0.2
        else:
            score -= 0.1
    
    # Temporal relationships should have temporal indicators
    if any(temporal_word in rel_type for temporal_word in ['during', 'by', 'in', 'temporal', 'timeline']):
        if context_analysis['temporal_indicators']:
            score += 0.2
        else:
            score -= 0.1
    
    # Quantitative relationships should have quantitative elements
    if any(quant_word in rel_type for quant_word in ['comprise', 'cover', 'measure', 'equal', '%']):
        if context_analysis['quantitative_elements']:
            score += 0.2
        else:
            score -= 0.1
    
    # Spatial relationships should have spatial indicators
    if any(spatial_word in rel_type for spatial_word in ['locate', 'within', 'encompass', 'adjacent']):
        if context_analysis['spatial_indicators']:
            score += 0.2
        else:
            score -= 0.1
    
    return min(max(score, 0.0), 1.0)  # Clamp between 0 and 1

def calculate_semantic_coherence(relation):
    """Calculate semantic coherence of the relationship."""
    
    score = 0.5  # Base score
    
    # Define domain-appropriate relationship patterns
    lulc_entity_types = ['forest', 'urban', 'agricultural', 'water', 'grass', 'built', 'city', 'area', 'land']
    process_types = ['expansion', 'growth', 'decline', 'conversion', 'transformation', 'development']
    temporal_types = ['2050', '2030', '2040', 'future', 'past', 'year', 'period']
    quantitative_types = ['%', 'hectare', 'km', 'acre']
    
    source = relation['source'].lower()
    target = relation['target'].lower()
    relationship = relation['relationship'].lower()
    
    # Check for semantically valid LULC relationships
    source_is_lulc = any(lulc_type in source for lulc_type in lulc_entity_types)
    target_is_lulc = any(lulc_type in target for lulc_type in lulc_entity_types)
    source_is_process = any(proc_type in source for proc_type in process_types)
    target_is_process = any(proc_type in target for proc_type in process_types)
    source_is_temporal = any(temp_type in source for temp_type in temporal_types)
    target_is_temporal = any(temp_type in target for temp_type in temporal_types)
    source_is_quant = any(quant_type in source for quant_type in quantitative_types)
    target_is_quant = any(quant_type in target for quant_type in quantitative_types)
    
    # Reward semantically coherent relationships
    coherent_patterns = [
        # LULC to LULC transformations
        (source_is_lulc and target_is_lulc and any(word in relationship for word in ['convert', 'transform', 'change', 'replace'])),
        # Process affecting LULC
        (source_is_process and target_is_lulc and any(word in relationship for word in ['affect', 'impact', 'cause', 'lead'])),
        # LULC quantified by percentages/areas
        (source_is_lulc and target_is_quant and any(word in relationship for word in ['cover', 'comprise', 'measure', 'equal'])),
        # Temporal context for processes
        (source_is_temporal and (target_is_process or target_is_lulc) and any(word in relationship for word in ['temporal', 'during', 'by'])),
        # Causal relationships
        (any(word in relationship for word in ['cause', 'lead', 'result']) and (source_is_lulc or source_is_process))
    ]
    
    if any(coherent_patterns):
        score += 0.3
    
    # Penalize likely incoherent relationships
    incoherent_patterns = [
        # Numbers causing abstract concepts (usually wrong)
        (source_is_quant and not target_is_lulc and 'cause' in relationship),
        # Temporal entities being locations
        (source_is_temporal and 'locate' in relationship)
    ]
    
    if any(incoherent_patterns):
        score -= 0.3
    
    return min(max(score, 0.0), 1.0)

def adjust_confidence_with_scores(original_confidence, overall_score):
    """Adjust confidence level based on quality scores."""
    
    confidence_mapping = {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3}
    reverse_mapping = {1: 'LOW', 2: 'MEDIUM', 3: 'HIGH'}
    
    current_level = confidence_mapping.get(original_confidence, 2)
    
    # Adjust based on overall score
    if overall_score >= 0.8:
        adjusted_level = min(current_level + 1, 3)
    elif overall_score >= 0.6:
        adjusted_level = current_level
    elif overall_score >= 0.4:
        adjusted_level = max(current_level - 1, 1)
    else:
        adjusted_level = 1
    
    return reverse_mapping[adjusted_level]

def calculate_extraction_quality(entities, relations, sentence, context_analysis):
    """Calculate comprehensive extraction quality metrics."""
    
    # Entity coverage score
    important_indicators = (
        context_analysis['lulc_indicators'] + 
        context_analysis['process_indicators'] + 
        context_analysis['quantitative_elements'] +
        context_analysis['temporal_indicators'][:2]  # Limit temporal to avoid over-counting
    )
    
    entity_texts_lower = [ent['text'].lower() for ent in entities]
    covered_indicators = sum(1 for indicator in important_indicators 
                           if any(indicator in entity_text for entity_text in entity_texts_lower))
    
    entity_coverage = covered_indicators / max(len(important_indicators), 1)
    
    # Relation diversity score
    relation_types = set(rel['relationship'] for rel in relations)
    expected_types = ['causal', 'temporal', 'quantitative', 'spatial', 'transformation']
    
    type_coverage = 0
    for rel_type in relation_types:
        rel_lower = rel_type.lower()
        if any(expected in rel_lower for expected in expected_types):
            type_coverage += 1
    
    relation_diversity = min(type_coverage / len(expected_types), 1.0)
    
    # Confidence distribution
    high_conf_relations = [r for r in relations if r.get('confidence') == 'HIGH']
    confidence_ratio = len(high_conf_relations) / max(len(relations), 1)
    
    # Context utilization score
    context_utilization = calculate_context_utilization(relations, context_analysis)
    
    # Overall quality score
    overall_quality = (
        entity_coverage * 0.25 +
        relation_diversity * 0.25 +
        confidence_ratio * 0.25 +
        context_utilization * 0.25
    )
    
    return {
        'overall_quality': overall_quality,
        'entity_coverage': entity_coverage,
        'relation_diversity': relation_diversity,
        'confidence_ratio': confidence_ratio,
        'context_utilization': context_utilization,
        'total_entities': len(entities),
        'total_relations': len(relations),
        'context_indicators_found': len(important_indicators),
        'context_indicators_covered': covered_indicators
    }

def calculate_context_utilization(relations, context_analysis):
    """Calculate how well the extraction utilized available context."""
    
    # Check if relations align with context indicators
    context_types = ['causal', 'temporal', 'quantitative', 'spatial', 'process']
    utilized_types = set()
    
    for relation in relations:
        rel_lower = relation['relationship'].lower()
        
        # Check which context types this relation utilizes
        if (any(word in rel_lower for word in ['cause', 'lead', 'result', 'trigger', 'drive']) and
            context_analysis['causal_indicators']):
            utilized_types.add('causal')
        
        if (any(word in rel_lower for word in ['during', 'by', 'temporal', 'timeline']) and
            context_analysis['temporal_indicators']):
            utilized_types.add('temporal')
        
        if (any(word in rel_lower for word in ['comprise', 'cover', 'measure', 'equal']) and
            context_analysis['quantitative_elements']):
            utilized_types.add('quantitative')
        
        if (any(word in rel_lower for word in ['locate', 'within', 'encompass']) and
            context_analysis['spatial_indicators']):
            utilized_types.add('spatial')
        
        if (any(word in rel_lower for word in ['expand', 'grow', 'convert', 'transform']) and
            context_analysis['process_indicators']):
            utilized_types.add('process')
    
    # Calculate utilization ratio
    available_types = []
    if context_analysis['causal_indicators']: available_types.append('causal')
    if context_analysis['temporal_indicators']: available_types.append('temporal')
    if context_analysis['quantitative_elements']: available_types.append('quantitative')
    if context_analysis['spatial_indicators']: available_types.append('spatial')
    if context_analysis['process_indicators']: available_types.append('process')
    
    utilization_ratio = len(utilized_types) / max(len(available_types), 1)
    
    return utilization_ratio

print("✅ Context validation and quality assessment functions loaded")

✅ Context validation and quality assessment functions loaded


In [8]:
def process_grammar_aware_relations(processed_sentences, model, tokenizer, limit=None):
    """Main pipeline for grammar-aware relation extraction."""
    
    if limit:
        processed_sentences = processed_sentences[:limit]
    
    print(f"🚀 Starting grammar-aware relation extraction for {len(processed_sentences)} sentences")
    
    results = []
    
    for i, sentence_data in enumerate(tqdm(processed_sentences, desc="Extracting relations")):
        
        if sentence_data.get('processing_failed'):
            results.append(sentence_data)  # Keep failed items
            continue
        
        try:
            print(f"\n🔍 Processing sentence {i+1}/{len(processed_sentences)}")
            print(f"Sentence: {sentence_data['original_sentence'][:80]}...")
            
            # Extract relations
            raw_relations = extract_grammar_relations(sentence_data, model, tokenizer)
            
            # Assess quality
            if raw_relations:
                assessed_relations = assess_grammar_relation_quality(raw_relations, sentence_data)
                filtered_results = filter_high_quality_relations(assessed_relations)
                
                final_relations = filtered_results['filtered']
            else:
                assessed_relations = []
                filtered_results = {'filtered': [], 'high_quality': [], 'medium_quality': [], 'low_quality': []}
                final_relations = []
            
            # Compile results
            result = {
                'id': sentence_data['id'],
                'original_sentence': sentence_data['original_sentence'],
                'original_entities': sentence_data['original_entities'],
                'enhanced_entities': sentence_data['enhanced_entities'],
                'grammar_analysis': sentence_data['grammar_analysis'],
                'raw_relations': raw_relations,
                'assessed_relations': assessed_relations,
                'final_relations': final_relations,
                'quality_breakdown': {
                    'high_quality_count': len(filtered_results['high_quality']),
                    'medium_quality_count': len(filtered_results['medium_quality']),
                    'low_quality_count': len(filtered_results['low_quality']),
                    'total_extracted': len(raw_relations),
                    'total_accepted': len(final_relations)
                },
                'processing_metadata': sentence_data['processing_metadata']
            }
            
            # Add grammar-specific metadata
            result['processing_metadata'].update({
                'relations_extracted': len(raw_relations),
                'relations_high_quality': len(filtered_results['high_quality']),
                'relations_accepted': len(final_relations),
                'average_relation_quality': np.mean([r['quality_scores']['overall_quality'] for r in assessed_relations]) if assessed_relations else 0,
                'grammar_method_used': True
            })
            
            results.append(result)
            
            # Progress summary
            print(f"   Relations: {len(raw_relations)} extracted → {len(final_relations)} accepted")
            if assessed_relations:
                avg_quality = np.mean([r['quality_scores']['overall_quality'] for r in assessed_relations])
                print(f"   Average quality: {avg_quality:.3f}")
            
        except Exception as e:
            print(f"❌ Error processing sentence {i}: {e}")
            
            # Add error result
            error_result = sentence_data.copy()
            error_result.update({
                'extraction_error': str(e),
                'extraction_failed': True,
                'final_relations': []
            })
            results.append(error_result)
    
    return results

# Execute grammar-aware processing
LIMIT_SENTENCES = 10  # Set to None for full processing

print("🚀 STARTING GRAMMAR-AWARE LULC RELATION EXTRACTION")
print("=" * 60)

grammar_results = process_grammar_aware_relations(
    processed_sentences, 
    model, 
    tokenizer, 
    limit=LIMIT_SENTENCES
)

# Calculate summary statistics
successful_results = [r for r in grammar_results if not r.get('processing_failed') and not r.get('extraction_failed')]

if successful_results:
    print(f"\n📊 GRAMMAR-AWARE EXTRACTION SUMMARY")
    print("=" * 50)
    
    total_sentences = len(grammar_results)
    successful_count = len(successful_results)
    
    # Entity statistics
    avg_original_entities = np.mean([r['processing_metadata']['original_entity_count'] for r in successful_results])
    avg_enhanced_entities = np.mean([r['processing_metadata']['enhanced_entity_count'] for r in successful_results])
    avg_discovered = np.mean([r['processing_metadata']['entities_discovered'] for r in successful_results])
    
    # Relation statistics
    avg_relations_extracted = np.mean([r['processing_metadata']['relations_extracted'] for r in successful_results])
    avg_relations_accepted = np.mean([r['processing_metadata']['relations_accepted'] for r in successful_results])
    avg_relation_quality = np.mean([r['processing_metadata']['average_relation_quality'] for r in successful_results])
    
    # Grammar statistics
    avg_grammatical_roles = np.mean([r['processing_metadata']['grammatical_roles_found'] for r in successful_results])
    avg_clauses = np.mean([r['processing_metadata']['clauses_found'] for r in successful_results])
    
    print(f"Success rate: {successful_count}/{total_sentences} ({successful_count/total_sentences*100:.1f}%)")
    print(f"")
    print(f"📋 ENTITY ANALYSIS:")
    print(f"   Original entities: {avg_original_entities:.1f} per sentence")
    print(f"   Enhanced entities: {avg_enhanced_entities:.1f} per sentence")
    print(f"   Entities discovered: {avg_discovered:.1f} per sentence")
    print(f"   Grammatical roles found: {avg_grammatical_roles:.1f} per sentence")
    print(f"")
    print(f"🔗 RELATION ANALYSIS:")
    print(f"   Relations extracted: {avg_relations_extracted:.1f} per sentence")
    print(f"   Relations accepted: {avg_relations_accepted:.1f} per sentence")
    print(f"   Average relation quality: {avg_relation_quality:.3f}")
    print(f"")
    print(f"📝 GRAMMAR ANALYSIS:")
    print(f"   Average clauses per sentence: {avg_clauses:.1f}")

print("✅ Grammar-aware processing complete")

🚀 STARTING GRAMMAR-AWARE LULC RELATION EXTRACTION
🚀 Starting grammar-aware relation extraction for 10 sentences


Extracting relations:   0%|          | 0/10 [00:00<?, ?it/s]


🔍 Processing sentence 1/10
Sentence: Simulation results reveal that the landscape of Thimphu city has changed conside...
❌ Error processing sentence 0: name 'extract_grammar_relations' is not defined

🔍 Processing sentence 2/10
Sentence: The study observed a significant increase (12.77%) in built-up area from 2002 (5...
❌ Error processing sentence 1: name 'extract_grammar_relations' is not defined

🔍 Processing sentence 3/10
Sentence: On the contrary, forest cover declined drastically (15.25%) followed by agricult...
❌ Error processing sentence 2: name 'extract_grammar_relations' is not defined

🔍 Processing sentence 4/10
Sentence: Rapid population growth triggered by rural urban migration coupled with hasty so...
❌ Error processing sentence 3: name 'extract_grammar_relations' is not defined

🔍 Processing sentence 5/10
Sentence: Under the business as usual scenario, prediction analysis for the year 2050 show...
❌ Error processing sentence 4: name 'extract_grammar_relations' is not def

In [9]:
def display_detailed_results(results, num_samples=3):
    """Display detailed results with grammar analysis."""
    
    successful_results = [r for r in results if not r.get('processing_failed') and not r.get('extraction_failed')]
    
    print(f"\n📝 DETAILED GRAMMAR-AWARE EXTRACTION RESULTS")
    print("=" * 70)
    
    for i, result in enumerate(successful_results[:num_samples]):
        print(f"\n🔍 RESULT {i+1}/{min(num_samples, len(successful_results))}")
        print("-" * 50)
        
        print(f"📄 Sentence: {result['original_sentence']}")
        print(f"📊 Length: {len(result['original_sentence'])} chars, {len(result['original_sentence'].split())} words")
        
        # Grammar analysis summary
        grammar = result['grammar_analysis']
        structure = grammar['sentence_structure']
        print(f"\n🔤 GRAMMAR ANALYSIS:")
        print(f"   Type: {structure.get('sentence_type', 'unknown')} | Complexity: {structure.get('complexity', 'unknown')}")
        print(f"   Passive: {structure.get('passive_voice', False)} | Negation: {structure.get('negation', False)}")
        print(f"   Clauses: {len(grammar['clauses'])}")
        print(f"   Subjects: {len(grammar['subjects'])}, Objects: {len(grammar['objects'])}, Predicates: {len(grammar['predicates'])}")
        
        # Entity enhancement
        print(f"\n🏷️  ENTITY ENHANCEMENT:")
        print(f"   Original: {len(result['original_entities'])} → Enhanced: {len(result['enhanced_entities'])} (+{result['processing_metadata']['entities_discovered']})")
        
        print(f"   📋 Enhanced Entities:")
        for j, entity in enumerate(result['enhanced_entities'][:8]):  # Show first 8
            role_info = entity.get('grammatical_role', {})
            role = role_info.get('role', 'unknown')
            
            discovered_tag = " [DISCOVERED]" if entity.get('discovery_method') else ""
            print(f"      [{j}] {entity['text']} ({entity['label']}) | Role: {role}{discovered_tag}")
        
        if len(result['enhanced_entities']) > 8:
            print(f"      ... and {len(result['enhanced_entities']) - 8} more entities")
        
        # Relations analysis
        final_relations = result['final_relations']
        print(f"\n🔗 RELATIONS EXTRACTED:")
        print(f"   Raw extracted: {len(result['raw_relations'])}")
        print(f"   Quality filtered: {len(final_relations)}")
        print(f"   Quality breakdown: {result['quality_breakdown']['high_quality_count']} high, {result['quality_breakdown']['medium_quality_count']} medium, {result['quality_breakdown']['low_quality_count']} low")
        
        if final_relations:
            print(f"   📋 Final Relations:")
            for relation in final_relations[:5]:  # Show first 5
                quality = relation['quality_scores']['overall_quality']
                grammar_just = relation.get('grammar_justification', 'N/A')[:40]
                
                print(f"      {relation['source']} --{relation['relationship']}--> {relation['target']}")
                print(f"         Confidence: {relation['confidence']} | Quality: {quality:.3f} | Grammar: {grammar_just}...")
        else:
            print(f"   ⚠️ No high-quality relations found")
        
        # Quality scores
        if result['assessed_relations']:
            qualities = [r['quality_scores']['overall_quality'] for r in result['assessed_relations']]
            grammar_scores = [r['quality_scores']['grammar_alignment'] for r in result['assessed_relations']]
            semantic_scores = [r['quality_scores']['semantic_coherence'] for r in result['assessed_relations']]
            
            print(f"\n📈 QUALITY METRICS:")
            print(f"   Overall Quality: {np.mean(qualities):.3f} (±{np.std(qualities):.3f})")
            print(f"   Grammar Alignment: {np.mean(grammar_scores):.3f}")
            print(f"   Semantic Coherence: {np.mean(semantic_scores):.3f}")

def create_grammar_analysis_visualization(results):
    """Create visualizations of grammar analysis results."""
    
    successful_results = [r for r in results if not r.get('processing_failed') and not r.get('extraction_failed')]
    
    if not successful_results:
        print("❌ No successful results to visualize")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Grammar-Aware LULC Extraction Analysis', fontsize=16, fontweight='bold')
    
    # 1. Entity Enhancement Distribution
    original_counts = [r['processing_metadata']['original_entity_count'] for r in successful_results]
    enhanced_counts = [r['processing_metadata']['enhanced_entity_count'] for r in successful_results]
    
    axes[0, 0].scatter(original_counts, enhanced_counts, alpha=0.6, color='blue')
    axes[0, 0].plot([0, max(original_counts)], [0, max(original_counts)], 'r--', alpha=0.5)
    axes[0, 0].set_xlabel('Original Entities')
    axes[0, 0].set_ylabel('Enhanced Entities')
    axes[0, 0].set_title('Entity Enhancement')
    
    # 2. Grammatical Roles Distribution
    roles_counts = [r['processing_metadata']['grammatical_roles_found'] for r in successful_results]
    axes[0, 1].hist(roles_counts, bins=10, color='green', alpha=0.7, edgecolor='black')
    axes[0, 1].set_xlabel('Grammatical Roles Found')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Grammatical Roles Distribution')
    
    # 3. Relation Quality Distribution
    all_qualities = []
    for result in successful_results:
        for relation in result['assessed_relations']:
            all_qualities.append(relation['quality_scores']['overall_quality'])
    
    if all_qualities:
        axes[0, 2].hist(all_qualities, bins=15, color='orange', alpha=0.7, edgecolor='black')
        axes[0, 2].set_xlabel('Relation Quality Score')
        axes[0, 2].set_ylabel('Frequency')
        axes[0, 2].set_title('Relation Quality Distribution')
        axes[0, 2].axvline(np.mean(all_qualities), color='red', linestyle='--', label=f'Mean: {np.mean(all_qualities):.3f}')
        axes[0, 2].legend()
    
    # 4. Relations Extracted vs Accepted
    extracted_counts = [r['processing_metadata']['relations_extracted'] for r in successful_results]
    accepted_counts = [r['processing_metadata']['relations_accepted'] for r in successful_results]
    
    axes[1, 0].scatter(extracted_counts, accepted_counts, alpha=0.6, color='purple')
    axes[1, 0].set_xlabel('Relations Extracted')
    axes[1, 0].set_ylabel('Relations Accepted')
    axes[1, 0].set_title('Relation Filtering Effectiveness')
    
    # 5. Sentence Complexity vs Relations
    clause_counts = [len(r['grammar_analysis']['clauses']) for r in successful_results]
    axes[1, 1].scatter(clause_counts, accepted_counts, alpha=0.6, color='red')
    axes[1, 1].set_xlabel('Number of Clauses')
    axes[1, 1].set_ylabel('Relations Accepted')
    axes[1, 1].set_title('Sentence Complexity vs Relations')
    
    # 6. Quality Component Breakdown
    if all_qualities:
        grammar_scores = []
        semantic_scores = []
        structural_scores = []
        
        for result in successful_results:
            for relation in result['assessed_relations']:
                grammar_scores.append(relation['quality_scores']['grammar_alignment'])
                semantic_scores.append(relation['quality_scores']['semantic_coherence'])
                structural_scores.append(relation['quality_scores']['structural_validity'])
        
        components = ['Grammar\nAlignment', 'Semantic\nCoherence', 'Structural\nValidity']
        means = [np.mean(grammar_scores), np.mean(semantic_scores), np.mean(structural_scores)]
        
        bars = axes[1, 2].bar(components, means, color=['blue', 'green', 'orange'], alpha=0.7)
        axes[1, 2].set_ylabel('Average Score')
        axes[1, 2].set_title('Quality Components')
        axes[1, 2].set_ylim(0, 1)
        
        for bar, value in zip(bars, means):
            axes[1, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                            f'{value:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

# Display results
display_detailed_results(grammar_results, num_samples=3)

# Create visualizations
print(f"\n📈 Creating visualizations...")
create_grammar_analysis_visualization(grammar_results)

print("✅ Results analysis complete")


📝 DETAILED GRAMMAR-AWARE EXTRACTION RESULTS

📈 Creating visualizations...
❌ No successful results to visualize
✅ Results analysis complete


In [10]:
def save_grammar_results(results, output_dir="grammar_extraction_results"):
    """Save all grammar-aware extraction results."""
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save complete results
    results_file = os.path.join(output_dir, f"grammar_results_{timestamp}.json")
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Complete results saved to: {results_file}")
    
    # Save grammar analysis only
    grammar_analysis_file = os.path.join(output_dir, f"grammar_analysis_{timestamp}.json")
    grammar_data = []
    for result in results:
        if not result.get('processing_failed') and 'grammar_analysis' in result:
            grammar_data.append({
                'id': result['id'],
                'sentence': result['original_sentence'],
                'grammar_analysis': result['grammar_analysis'],
                'entities_enhanced': result['enhanced_entities']
            })
    
    with open(grammar_analysis_file, 'w', encoding='utf-8') as f:
        json.dump(grammar_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Grammar analysis saved to: {grammar_analysis_file}")
    
    # Save final relations for Label Studio
    relations_file = os.path.join(output_dir, f"grammar_relations_{timestamp}.json")
    relations_data = []
    for result in results:
        if not result.get('processing_failed') and 'final_relations' in result:
            relations_data.append({
                'id': result['id'],
                'sentence': result['original_sentence'],
                'entities': result['enhanced_entities'],
                'relations': result['final_relations'],
                'quality_metrics': {
                    'total_extracted': len(result.get('raw_relations', [])),
                    'total_accepted': len(result['final_relations']),
                    'average_quality': result['processing_metadata'].get('average_relation_quality', 0)
                }
            })
    
    with open(relations_file, 'w', encoding='utf-8') as f:
        json.dump(relations_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Relations data saved to: {relations_file}")
    
    # Generate summary report
    summary_file = os.path.join(output_dir, f"grammar_summary_{timestamp}.txt")
    generate_summary_report(results, summary_file)
    
    return results_file, grammar_analysis_file, relations_file, summary_file

def generate_summary_report(results, output_file):
    """Generate a comprehensive summary report."""
    
    successful_results = [r for r in results if not r.get('processing_failed') and not r.get('extraction_failed')]
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("GRAMMAR-AWARE LULC RELATION EXTRACTION SUMMARY REPORT\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        # Overall statistics
        f.write("OVERALL STATISTICS:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Total sentences processed: {len(results)}\n")
        f.write(f"Successfully processed: {len(successful_results)}\n")
        f.write(f"Success rate: {len(successful_results)/len(results)*100:.1f}%\n\n")
        
        if successful_results:
            # Entity statistics
            f.write("ENTITY ANALYSIS:\n")
            f.write("-" * 15 + "\n")
            orig_entities = [r['processing_metadata']['original_entity_count'] for r in successful_results]
            enh_entities = [r['processing_metadata']['enhanced_entity_count'] for r in successful_results]
            discovered = [r['processing_metadata']['entities_discovered'] for r in successful_results]
            roles = [r['processing_metadata']['grammatical_roles_found'] for r in successful_results]
            
            f.write(f"Average original entities per sentence: {np.mean(orig_entities):.2f}\n")
            f.write(f"Average enhanced entities per sentence: {np.mean(enh_entities):.2f}\n")
            f.write(f"Average entities discovered per sentence: {np.mean(discovered):.2f}\n")
            f.write(f"Average grammatical roles found: {np.mean(roles):.2f}\n")
            f.write(f"Total entities discovered: {sum(discovered)}\n\n")
            
            # Relation statistics
            f.write("RELATION ANALYSIS:\n")
            f.write("-" * 17 + "\n")
            extracted = [r['processing_metadata']['relations_extracted'] for r in successful_results]
            accepted = [r['processing_metadata']['relations_accepted'] for r in successful_results]
            quality = [r['processing_metadata']['average_relation_quality'] for r in successful_results]
            
            f.write(f"Average relations extracted per sentence: {np.mean(extracted):.2f}\n")
            f.write(f"Average relations accepted per sentence: {np.mean(accepted):.2f}\n")
            f.write(f"Average relation quality score: {np.mean(quality):.3f}\n")
            f.write(f"Total relations extracted: {sum(extracted)}\n")
            f.write(f"Total relations accepted: {sum(accepted)}\n")
            f.write(f"Acceptance rate: {sum(accepted)/sum(extracted)*100:.1f}%\n\n")
            
            # Grammar analysis
            f.write("GRAMMAR ANALYSIS:\n")
            f.write("-" * 16 + "\n")
            clauses = [len(r['grammar_analysis']['clauses']) for r in successful_results]
            complexity_counts = Counter()
            sentence_types = Counter()
            
            for result in successful_results:
                structure = result['grammar_analysis']['sentence_structure']
                complexity_counts[structure.get('complexity', 'unknown')] += 1
                sentence_types[structure.get('sentence_type', 'unknown')] += 1
            
            f.write(f"Average clauses per sentence: {np.mean(clauses):.2f}\n")
            f.write(f"Sentence complexity distribution:\n")
            for complexity, count in complexity_counts.items():
                f.write(f"  {complexity}: {count} ({count/len(successful_results)*100:.1f}%)\n")
            f.write(f"Sentence type distribution:\n")
            for stype, count in sentence_types.items():
                f.write(f"  {stype}: {count} ({count/len(successful_results)*100:.1f}%)\n")
    
    print(f"✅ Summary report saved to: {output_file}")

# Save all results
output_files = save_grammar_results(grammar_results)

print(f"\n💾 All results saved:")
for file_path in output_files:
    print(f"   - {file_path}")

print(f"\n🎉 Grammar-Aware LULC Relation Extraction Complete!")
print(f"📊 Check the saved files and visualizations for detailed analysis.")

✅ Complete results saved to: grammar_extraction_results/grammar_results_20250627_143228.json
✅ Grammar analysis saved to: grammar_extraction_results/grammar_analysis_20250627_143228.json
✅ Relations data saved to: grammar_extraction_results/grammar_relations_20250627_143228.json
✅ Summary report saved to: grammar_extraction_results/grammar_summary_20250627_143228.txt

💾 All results saved:
   - grammar_extraction_results/grammar_results_20250627_143228.json
   - grammar_extraction_results/grammar_analysis_20250627_143228.json
   - grammar_extraction_results/grammar_relations_20250627_143228.json
   - grammar_extraction_results/grammar_summary_20250627_143228.txt

🎉 Grammar-Aware LULC Relation Extraction Complete!
📊 Check the saved files and visualizations for detailed analysis.


In [11]:
def convert_to_label_studio_format(grammar_results, output_file="grammar_label_studio.json"):
    """Convert grammar-aware extraction results to Label Studio format."""
    
    label_studio_data = []
    
    for result in grammar_results:
        if result.get('processing_failed') or result.get('extraction_failed'):
            continue
        
        sentence = result['original_sentence']
        entities = result['enhanced_entities']
        relations = result['final_relations']
        
        # Create Label Studio task
        task = {
            "id": result['id'],
            "data": {
                "text": sentence
            },
            "annotations": [],
            "predictions": [
                {
                    "id": f"prediction_{result['id']}",
                    "result": [],
                    "model_version": "grammar-aware-mistral-v1.0",
                    "score": result.get('quality_metrics', {}).get('average_quality', 0),
                    "created_ago": "now",
                    "cluster": None,
                    "neighbors": None,
                    "mislabeling": 0,
                    "created_at": datetime.now().isoformat(),
                    "updated_at": datetime.now().isoformat(),
                    "model": "grammar_aware_extraction",
                    "task": result['id'],
                    "project": 1
                }
            ]
        }
        
        # Add entities to predictions
        entity_id_map = {}
        for i, entity in enumerate(entities):
            entity_id = f"grammar_ent_{result['id']}_{i}"
            entity_id_map[entity['text']] = entity_id
            
            # Calculate confidence score
            confidence_score = calculate_entity_confidence(entity)
            
            # Create entity annotation
            entity_annotation = {
                "value": {
                    "start": entity.get('start_char', 0),
                    "end": entity.get('end_char', len(entity['text'])),
                    "text": entity['text'],
                    "labels": [entity['label']]
                },
                "id": entity_id,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "score": confidence_score
            }
            
            # Add grammar metadata
            if entity.get('grammatical_role'):
                entity_annotation["meta"] = {
                    "grammatical_role": entity['grammatical_role'].get('role', 'unknown'),
                    "syntactic_pos": entity.get('syntactic_info', {}).get('pos', 'unknown'),
                    "discovery_method": entity.get('discovery_method', 'original_ner'),
                    "confidence_level": entity.get('confidence', 'HIGH')
                }
            
            task["predictions"][0]["result"].append(entity_annotation)
        
        # Add relations to predictions
        for j, relation in enumerate(relations):
            # Find entity IDs for source and target
            source_id = entity_id_map.get(relation['source'])
            target_id = entity_id_map.get(relation['target'])
            
            if source_id and target_id:
                relation_annotation = {
                    "from_id": source_id,
                    "to_id": target_id,
                    "type": "relation",
                    "direction": "right",
                    "labels": [relation['relationship']],
                    "from_name": "relation",
                    "to_name": "label",
                    "score": convert_confidence_to_score(relation['confidence']),
                    "meta": {
                        "confidence": relation['confidence'],
                        "quality_score": relation.get('quality_scores', {}).get('overall_quality', 0),
                        "grammar_justification": relation.get('grammar_justification', 'N/A'),
                        "extraction_method": "grammar_aware",
                        "semantic_coherence": relation.get('quality_scores', {}).get('semantic_coherence', 0),
                        "grammar_alignment": relation.get('quality_scores', {}).get('grammar_alignment', 0)
                    }
                }
                
                task["predictions"][0]["result"].append(relation_annotation)
        
        label_studio_data.append(task)
    
    # Save to file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(label_studio_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Label Studio data saved to: {output_file}")
    print(f"📊 Exported {len(label_studio_data)} tasks")
    
    return output_file

def calculate_entity_confidence(entity):
    """Calculate confidence score for Label Studio (0.0 to 1.0)."""
    
    confidence_map = {
        'HIGH': 0.95,
        'MEDIUM': 0.75,
        'LOW': 0.55
    }
    
    base_confidence = confidence_map.get(entity.get('confidence', 'MEDIUM'), 0.75)
    
    # Adjust based on discovery method
    if entity.get('discovery_method'):
        if entity['discovery_method'] == 'grammar_subject':
            base_confidence += 0.05  # Grammar subjects are usually important
        elif entity['discovery_method'] == 'grammar_object':
            base_confidence += 0.03
        elif entity['discovery_method'] == 'grammar_noun_phrase':
            base_confidence -= 0.05  # Noun phrases might be less precise
    
    # Adjust based on grammatical role
    role = entity.get('grammatical_role', {}).get('role', 'unknown')
    if role in ['subject', 'object']:
        base_confidence += 0.05
    elif role == 'predicate':
        base_confidence += 0.03
    elif role == 'modifier':
        base_confidence -= 0.02
    
    return min(max(base_confidence, 0.1), 1.0)  # Clamp between 0.1 and 1.0

def convert_confidence_to_score(confidence_text):
    """Convert text confidence to numerical score."""
    
    confidence_map = {
        'HIGH': 0.9,
        'MEDIUM': 0.7,
        'LOW': 0.5
    }
    
    return confidence_map.get(confidence_text, 0.7)

def create_grammar_label_studio_config():
    """Create Label Studio configuration for grammar-aware annotations."""
    
    # Extract all unique entity types and relation types from results
    all_entity_types = set()
    all_relation_types = set()
    
    for result in grammar_results:
        if result.get('processing_failed') or result.get('extraction_failed'):
            continue
        
        for entity in result.get('enhanced_entities', []):
            all_entity_types.add(entity['label'])
        
        for relation in result.get('final_relations', []):
            all_relation_types.add(relation['relationship'])
    
    # Enhanced colors for better visualization
    colors = [
        "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", 
        "#FF9FF3", "#A55EEA", "#54A0FF", "#5F27CD", "#00D2D3",
        "#FF9F43", "#10AC84", "#EE5A24", "#0098C7", "#8395A7",
        "#FD79A8", "#FDCB6E", "#6C5CE7", "#A29BFE", "#74B9FF",
        "#E17055", "#81ECEC", "#55A3FF", "#FD79A8", "#FDCB6E"
    ]
    
    # Build entity labels with descriptions
    entity_labels = ""
    entity_descriptions = {
        'LOC': 'Location/Place names',
        'LULC': 'Land Use/Land Cover types',
        'CHANGE': 'Change/Transformation processes',
        'DATE': 'Dates and time periods',
        'DISCOVERED_SUBJECT': 'Grammar-discovered subjects',
        'DISCOVERED_OBJECT': 'Grammar-discovered objects',
        'DISCOVERED_LULC': 'Grammar-discovered LULC terms'
    }
    
    for i, entity_type in enumerate(sorted(all_entity_types)):
        color = colors[i % len(colors)]
        description = entity_descriptions.get(entity_type, 'Entity type')
        safe_entity = entity_type.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        
        entity_labels += f'    <Label value="{safe_entity}" background="{color}" description="{description}"/>\n'
    
    # Build relation labels with categories
    relation_labels = ""
    relation_categories = {
        'causal': ['causes', 'leads_to', 'results_in', 'triggers', 'drives', 'affects'],
        'temporal': ['occurs_during', 'happens_by', 'continues_into', 'happened_during'],
        'spatial': ['located_in', 'is_located_in', 'has_undergone_change', 'is_in'],
        'informational': ['reveals_information_about', 'reveals', 'shows'],
        'general': []
    }
    
    # Categorize relations
    categorized_relations = {}
    for relation_type in sorted(all_relation_types):
        category = 'general'
        for cat, keywords in relation_categories.items():
            if any(keyword in relation_type.lower() for keyword in keywords):
                category = cat
                break
        
        if category not in categorized_relations:
            categorized_relations[category] = []
        categorized_relations[category].append(relation_type)
    
    # Build relation XML
    color_idx = 0
    for category, relations in categorized_relations.items():
        for relation_type in relations:
            color = colors[color_idx % len(colors)]
            safe_relation = relation_type.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
            relation_labels += f'    <Relation value="{safe_relation}" background="{color}" description="{category} relationship"/>\n'
            color_idx += 1
    
    # Create comprehensive XML configuration
    config_xml = f"""<View>
  <Header value="Grammar-Aware LULC Entity and Relation Annotation"/>
  
  <Text name="text" value="$text"/>
  
  <!-- Entity Labels -->
  <Labels name="label" toName="text" showInline="true">
{entity_labels}  </Labels>
  
  <!-- Relation Labels -->
  <Relations name="relation" toName="label">
{relation_labels}  </Relations>
  
  <!-- Quality Assessment -->
  <Choices name="overall_quality" toName="text" choice="single" showInline="false">
    <Choice value="Excellent" background="#00ff00"/>
    <Choice value="Good" background="#90EE90"/>
    <Choice value="Fair" background="#FFD700"/>
    <Choice value="Poor" background="#FF6347"/>
    <Choice value="Very Poor" background="#FF0000"/>
  </Choices>
  
  <!-- Grammar Quality -->
  <Choices name="grammar_quality" toName="text" choice="single" showInline="false">
    <Choice value="Grammar Analysis Accurate"/>
    <Choice value="Grammar Analysis Partial"/>
    <Choice value="Grammar Analysis Incorrect"/>
  </Choices>
  
  <!-- Relation Quality -->
  <Choices name="relation_quality" toName="text" choice="single" showInline="false">
    <Choice value="Relations Semantically Correct"/>
    <Choice value="Relations Partially Correct"/>
    <Choice value="Relations Need Major Correction"/>
  </Choices>
  
  <!-- Comments -->
  <TextArea name="comments" toName="text" placeholder="Additional comments about the annotation quality..." rows="3" showInline="false"/>
  
</View>"""
    
    return config_xml

def save_label_studio_config(config_xml, output_file="grammar_label_studio_config.xml"):
    """Save Label Studio configuration to file."""
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(config_xml)
    
    print(f"✅ Label Studio configuration saved to: {output_file}")
    return output_file

def create_label_studio_project_info(label_studio_file, config_file):
    """Create project information file for Label Studio setup."""
    
    project_info = {
        "project_name": "Grammar-Aware LULC Relation Extraction",
        "description": "Annotation and validation of grammar-aware entity-relation extraction for LULC analysis",
        "data_file": label_studio_file,
        "config_file": config_file,
        "setup_instructions": [
            "1. Create new project in Label Studio",
            "2. Upload the configuration from: " + config_file,
            "3. Import data from: " + label_studio_file,
            "4. Start annotating and validating the predictions",
            "5. Focus on relation accuracy and grammatical correctness"
        ],
        "annotation_guidelines": {
            "entities": {
                "verify_boundaries": "Check if entity boundaries are correct",
                "verify_labels": "Confirm entity labels match the content",
                "grammar_discoveries": "Validate grammar-discovered entities"
            },
            "relations": {
                "semantic_correctness": "Ensure relations make semantic sense",
                "direction_accuracy": "Check if relation direction is correct",
                "grammar_alignment": "Verify relations align with sentence grammar"
            },
            "quality_assessment": {
                "overall": "Rate the overall extraction quality",
                "grammar": "Assess grammar analysis accuracy",
                "relations": "Evaluate relation semantic correctness"
            }
        },
        "entity_types": list(set(entity['label'] for result in grammar_results 
                                if not result.get('processing_failed') 
                                for entity in result.get('enhanced_entities', []))),
        "relation_types": list(set(relation['relationship'] for result in grammar_results 
                                  if not result.get('processing_failed') 
                                  for relation in result.get('final_relations', [])))
    }
    
    info_file = "label_studio_project_info.json"
    with open(info_file, 'w', encoding='utf-8') as f:
        json.dump(project_info, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Project information saved to: {info_file}")
    return info_file

def generate_label_studio_statistics(grammar_results):
    """Generate statistics for Label Studio export."""
    
    successful_results = [r for r in grammar_results if not r.get('processing_failed') and not r.get('extraction_failed')]
    
    stats = {
        "total_tasks": len(successful_results),
        "total_entities": sum(len(r['enhanced_entities']) for r in successful_results),
        "total_relations": sum(len(r['final_relations']) for r in successful_results),
        "entity_types": {},
        "relation_types": {},
        "discovery_methods": {},
        "quality_distribution": {"high": 0, "medium": 0, "low": 0}
    }
    
    for result in successful_results:
        # Entity type distribution
        for entity in result['enhanced_entities']:
            label = entity['label']
            stats['entity_types'][label] = stats['entity_types'].get(label, 0) + 1
            
            # Discovery method distribution
            method = entity.get('discovery_method', 'original_ner')
            stats['discovery_methods'][method] = stats['discovery_methods'].get(method, 0) + 1
        
        # Relation type distribution
        for relation in result['final_relations']:
            rel_type = relation['relationship']
            stats['relation_types'][rel_type] = stats['relation_types'].get(rel_type, 0) + 1
        
        # Quality distribution
        avg_quality = result.get('quality_metrics', {}).get('average_quality', 0)
        if avg_quality >= 0.7:
            stats['quality_distribution']['high'] += 1
        elif avg_quality >= 0.5:
            stats['quality_distribution']['medium'] += 1
        else:
            stats['quality_distribution']['low'] += 1
    
    return stats

# Execute Label Studio export
print("🏷️ CREATING LABEL STUDIO EXPORT")
print("=" * 50)

# Convert to Label Studio format
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
label_studio_file = f"grammar_label_studio_{timestamp}.json"
convert_to_label_studio_format(grammar_results, label_studio_file)

# Create and save configuration
config_xml = create_grammar_label_studio_config()
config_file = f"grammar_label_studio_config_{timestamp}.xml"
save_label_studio_config(config_xml, config_file)

# Create project information
project_info_file = create_label_studio_project_info(label_studio_file, config_file)

# Generate statistics
stats = generate_label_studio_statistics(grammar_results)

print(f"\n📊 LABEL STUDIO EXPORT STATISTICS:")
print(f"Total tasks: {stats['total_tasks']}")
print(f"Total entities: {stats['total_entities']}")
print(f"Total relations: {stats['total_relations']}")
print(f"Entity types: {len(stats['entity_types'])}")
print(f"Relation types: {len(stats['relation_types'])}")

print(f"\n📋 Entity Type Distribution:")
for entity_type, count in sorted(stats['entity_types'].items(), key=lambda x: x[1], reverse=True):
    percentage = (count / stats['total_entities']) * 100
    print(f"  {entity_type}: {count} ({percentage:.1f}%)")

print(f"\n🔗 Relation Type Distribution:")
for rel_type, count in sorted(stats['relation_types'].items(), key=lambda x: x[1], reverse=True)[:10]:
    percentage = (count / stats['total_relations']) * 100
    print(f"  {rel_type}: {count} ({percentage:.1f}%)")

print(f"\n📈 Quality Distribution:")
for quality, count in stats['quality_distribution'].items():
    percentage = (count / stats['total_tasks']) * 100
    print(f"  {quality} quality: {count} ({percentage:.1f}%)")

print(f"\n💾 FILES CREATED:")
print(f"  📄 Label Studio data: {label_studio_file}")
print(f"  ⚙️  Configuration: {config_file}")
print(f"  📋 Project info: {project_info_file}")

print(f"\n🚀 LABEL STUDIO SETUP INSTRUCTIONS:")
print(f"1. Open Label Studio")
print(f"2. Create new project")
print(f"3. Upload configuration from: {config_file}")
print(f"4. Import data from: {label_studio_file}")
print(f"5. Start annotation/validation")

print(f"\n✅ Label Studio export complete!")

🏷️ CREATING LABEL STUDIO EXPORT
✅ Label Studio data saved to: grammar_label_studio_20250627_143229.json
📊 Exported 0 tasks
✅ Label Studio configuration saved to: grammar_label_studio_config_20250627_143229.xml
✅ Project information saved to: label_studio_project_info.json

📊 LABEL STUDIO EXPORT STATISTICS:
Total tasks: 0
Total entities: 0
Total relations: 0
Entity types: 0
Relation types: 0

📋 Entity Type Distribution:

🔗 Relation Type Distribution:

📈 Quality Distribution:


ZeroDivisionError: division by zero